<a href="https://colab.research.google.com/github/pranjalgahlot123/Geo-Tagged-Plastic-Waste-Reporting-and-Alert-Management-System/blob/main/Geo_Tagged_Plastic_Waste_Reporting_and_Alert_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
# Cell 1: Analyze the actual folder structure
import os

DATASET_PATH = "/content/drive/MyDrive/plastic waste"

print("Analyzing folder structure...")
print("\n📁 Folder hierarchy (first 3 levels):")
!find "{DATASET_PATH}" -type d -maxdepth 3 | head -30

print("\n📄 Sample image paths (first 20):")
!find "{DATASET_PATH}" -type f -name "*.jpg" -o -name "*.png" -o -name "*.jpeg" | head -20

print("\n🔍 Looking for any subfolder names that might indicate categories:")
!find "{DATASET_PATH}" -type d | sed 's|.*/||' | sort | uniq -c | sort -rn | head -20

Analyzing folder structure...

📁 Folder hierarchy (first 3 levels):
find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/content/drive/MyDrive/plastic waste
/content/drive/MyDrive/plastic waste/archive
/content/drive/MyDrive/plastic waste/archive/original
/content/drive/MyDrive/plastic waste/archive/original/battery
/content/drive/MyDrive/plastic waste/archive/original/clothes
/content/drive/MyDrive/plastic waste/archive/original/biological
/content/drive/MyDrive/plastic waste/archive/original/shoes
/content/drive/MyDrive/plastic waste/archive/original/glass
/content/drive/MyDrive/plastic waste/archive/original/metal
/content/drive/MyDrive/plastic waste/archive/original/plastic
/content/drive/MyDrive/plastic waste/archive/original/trash
/content/drive/MyDrive/plastic waste

In [51]:
# Cell 1: Load and categorize images correctly
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to dataset
DATASET_PATH = "/content/drive/MyDrive/plastic waste"

# Define categories
PLASTIC_CATEGORIES = ['plastic']
NON_PLASTIC_CATEGORIES = ['paper', 'metal', 'glass', 'battery', 'biological', 'clothes', 'cardboard', 'trash', 'shoes']

print("=" * 60)
print("CATEGORIZATION MAPPING")
print("=" * 60)
print(f"Plastic categories: {PLASTIC_CATEGORIES}")
print(f"Non-plastic categories: {NON_PLASTIC_CATEGORIES}")

# Load DINO model
print("\n🔄 Loading DINOv2 model...")
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
dino_model = AutoModel.from_pretrained('facebook/dinov2-small').to(device)
dino_model.eval()
print("✅ DINOv2 model loaded!")

# Custom Dataset class
class PlasticDataset(Dataset):
    def __init__(self, image_paths, labels, processor):
        self.image_paths = image_paths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            inputs = self.processor(images=image, return_tensors="pt")
            return inputs['pixel_values'].squeeze(0), torch.tensor(self.labels[idx], dtype=torch.float32)
        except Exception as e:
            print(f"Warning: Could not load {self.image_paths[idx]}: {e}")
            return torch.zeros(3, 224, 224), torch.tensor(0.0, dtype=torch.float32)

# Scan and categorize images
print("\n📁 Scanning and categorizing images...")
plastic_images = []
non_plastic_images = []

for root, dirs, files in os.walk(DATASET_PATH):
    # Get the folder name
    folder_name = os.path.basename(root).lower()

    for file in files:
        if file.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp')):
            full_path = os.path.join(root, file)

            # Check if in plastic category
            if folder_name in PLASTIC_CATEGORIES:
                plastic_images.append(full_path)
            # Check if in non-plastic category
            elif folder_name in NON_PLASTIC_CATEGORIES:
                non_plastic_images.append(full_path)

print(f"\n📊 Dataset Statistics:")
print(f"Plastic images: {len(plastic_images)}")
print(f"Non-plastic images: {len(non_plastic_images)}")
print(f"Total images: {len(plastic_images) + len(non_plastic_images)}")

if len(plastic_images) == 0 or len(non_plastic_images) == 0:
    print("\n⚠️ Warning: Missing one category. Let me show you what folders were found:")
    !find "{DATASET_PATH}" -type d -maxdepth 2 | tail -20
else:
    print("\n✅ Categories found successfully!")

Using device: cpu
CATEGORIZATION MAPPING
Plastic categories: ['plastic']
Non-plastic categories: ['paper', 'metal', 'glass', 'battery', 'biological', 'clothes', 'cardboard', 'trash', 'shoes']

🔄 Loading DINOv2 model...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

✅ DINOv2 model loaded!

📁 Scanning and categorizing images...

📊 Dataset Statistics:
Plastic images: 4791
Non-plastic images: 32020
Total images: 36811

✅ Categories found successfully!


In [36]:
# Cell 2: Balance and prepare data
if len(plastic_images) > 0 and len(non_plastic_images) > 0:
    # Balance the dataset (optional - use all data or balance)
    use_balancing = True  # Set to False to use all data

    if use_balancing:
        # Balance to the smaller class size
        min_size = min(len(plastic_images), len(non_plastic_images))
        print(f"\nBalancing dataset to {min_size} images per class...")

        import random
        random.seed(42)
        plastic_balanced = random.sample(plastic_images, min_size)
        non_plastic_balanced = random.sample(non_plastic_images, min_size)

        all_images = plastic_balanced + non_plastic_balanced
        all_labels = [1] * min_size + [0] * min_size
    else:
        # Use all images
        print(f"\nUsing all images...")
        all_images = plastic_images + non_plastic_images
        all_labels = [1] * len(plastic_images) + [0] * len(non_plastic_images)

    print(f"Total images: {len(all_images)}")
    print(f"Plastic: {sum(all_labels)}")
    print(f"Non-plastic: {len(all_labels) - sum(all_labels)}")

    # Split into train and validation
    train_images, val_images, train_labels, val_labels = train_test_split(
        all_images, all_labels, test_size=0.2, stratify=all_labels, random_state=42
    )

    print(f"\nTraining samples: {len(train_images)}")
    print(f"Validation samples: {len(val_images)}")

    # Create datasets
    train_dataset = PlasticDataset(train_images, train_labels, processor)
    val_dataset = PlasticDataset(val_images, val_labels, processor)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

    print("\n✅ Data loaders created!")
else:
    print("❌ Cannot proceed - missing one or both categories")


Balancing dataset to 4791 images per class...
Total images: 9582
Plastic: 4791
Non-plastic: 4791

Training samples: 7665
Validation samples: 1917

✅ Data loaders created!


Optimize DataLoader


In [38]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,              # 🔥 Increase batch size (if GPU allows)
    shuffle=True,
    num_workers=4,              # 🔥 Parallel loading
    pin_memory=True,            # 🔥 Faster GPU transfer
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

Enable cuDNN Benchmark

In [39]:
torch.backends.cudnn.benchmark = True

Use Channels Last (Advanced Boost)

In [40]:
dino_model = dino_model.to(device).to(memory_format=torch.channels_last)

In [43]:
from PIL import Image
from torch.utils.data import Dataset

class PlasticDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        # 🔥 Load image properly
        image = Image.open(img_path).convert("RGB")

        # 🔥 Apply transforms
        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]

        return image, label   # ✅ returns tensor, not string

In [44]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),   # 🔥 MUST HAVE
])

In [45]:
images, labels = next(iter(train_loader))

print(type(images))

<class 'torch.Tensor'>


In [46]:
images = images.to(device, non_blocking=True)
images = images.to(memory_format=torch.channels_last)

In [47]:
# Cell 3: FAST DINO Feature Extraction

def extract_features_fast(data_loader, name):
    features = []
    labels = []

    print(f"\n🚀 Extracting features from {name} (FAST MODE)...")

    dino_model.eval()  # ✅ Important

    with torch.no_grad():
        for i, (images, lbls) in enumerate(data_loader):

            # ✅ Ensure tensor (safety)
            if isinstance(images, list):
                images = torch.stack(images)

            # 🚀 Fast GPU transfer
            images = images.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)

            # 🚀 Mixed precision (BIG SPEED BOOST)
            with torch.cuda.amp.autocast():
                outputs = dino_model(images)
                cls_features = outputs.last_hidden_state[:, 0, :]

            features.append(cls_features.cpu())
            labels.append(lbls.cpu())

            if (i + 1) % 20 == 0:
                print(f"⚡ {name}: {i+1} batches done")

    return torch.cat(features), torch.cat(labels)


# 🔥 Run fast extraction
train_features, train_labels_tensor = extract_features_fast(train_loader, "training set")
val_features, val_labels_tensor = extract_features_fast(val_loader, "validation set")

print("\n✅ FAST extraction completed!")
print(f"Train shape: {train_features.shape}")
print(f"Val shape: {val_features.shape}")


🚀 Extracting features from training set (FAST MODE)...
⚡ training set: 20 batches done
⚡ training set: 40 batches done
⚡ training set: 60 batches done
⚡ training set: 80 batches done
⚡ training set: 100 batches done
⚡ training set: 120 batches done

🚀 Extracting features from validation set (FAST MODE)...
⚡ validation set: 20 batches done

✅ FAST extraction completed!
Train shape: torch.Size([7665, 384])
Val shape: torch.Size([1917, 384])


In [48]:
# Cell 4: Train classifier on DINO features
class DINOClassifier(nn.Module):
    def __init__(self, input_dim=384):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.classifier(x)

# Initialize classifier
classifier = DINOClassifier().to(device)

# Calculate class weights for imbalance (though we balanced, keep for safety)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Use the balanced labels from your dataset
all_labels_np = np.array([1]*4791 + [0]*4791)  # 4791 plastic, 4791 non-plastic
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=all_labels_np)
pos_weight = torch.tensor([class_weights[1] / class_weights[0]]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

# Training
print("\n🚀 Starting training...")
print("=" * 50)
best_val_acc = 0
patience_counter = 0
early_stop_patience = 5

for epoch in range(20):
    # Training phase
    classifier.train()
    train_correct = 0
    train_total = 0
    train_loss = 0

    # Shuffle features
    indices = torch.randperm(len(train_features))
    shuffled_features = train_features[indices]
    shuffled_labels = train_labels_tensor[indices]

    for i in range(0, len(shuffled_features), 32):
        batch_features = shuffled_features[i:i+32].to(device)
        batch_labels = shuffled_labels[i:i+32].to(device)

        optimizer.zero_grad()
        outputs = classifier(batch_features).squeeze()
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        train_correct += (predictions == batch_labels).sum().item()
        train_total += batch_labels.size(0)

    train_acc = train_correct / train_total
    avg_train_loss = train_loss / (len(shuffled_features) / 32)

    # Validation phase
    classifier.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():
        for i in range(0, len(val_features), 32):
            batch_features = val_features[i:i+32].to(device)
            batch_labels = val_labels_tensor[i:i+32].to(device)
            outputs = classifier(batch_features).squeeze()
            loss = criterion(outputs, batch_labels)

            val_loss += loss.item()
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (predictions == batch_labels).sum().item()
            val_total += batch_labels.size(0)

    val_acc = val_correct / val_total
    avg_val_loss = val_loss / (len(val_features) / 32)

    # Learning rate scheduling
    scheduler.step(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), "/content/plastic_classifier_best.pth")
        print(f"✅ Epoch {epoch+1:2d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {avg_train_loss:.4f} | ✓ NEW BEST")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"   Epoch {epoch+1:2d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {avg_train_loss:.4f} | Patience: {patience_counter}/{early_stop_patience}")

        if patience_counter >= early_stop_patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

print(f"\n🏆 Best validation accuracy: {best_val_acc:.4f}")
print("✅ Model saved as 'plastic_classifier_best.pth'")


🚀 Starting training...
✅ Epoch  1 | Train Acc: 0.9539 | Val Acc: 0.9781 | Loss: 0.1212 | ✓ NEW BEST
✅ Epoch  2 | Train Acc: 0.9813 | Val Acc: 0.9786 | Loss: 0.0518 | ✓ NEW BEST
✅ Epoch  3 | Train Acc: 0.9860 | Val Acc: 0.9849 | Loss: 0.0390 | ✓ NEW BEST
✅ Epoch  4 | Train Acc: 0.9897 | Val Acc: 0.9896 | Loss: 0.0254 | ✓ NEW BEST
   Epoch  5 | Train Acc: 0.9961 | Val Acc: 0.9885 | Loss: 0.0149 | Patience: 1/5
   Epoch  6 | Train Acc: 0.9961 | Val Acc: 0.9854 | Loss: 0.0118 | Patience: 2/5
   Epoch  7 | Train Acc: 0.9952 | Val Acc: 0.9880 | Loss: 0.0157 | Patience: 3/5
   Epoch  8 | Train Acc: 0.9966 | Val Acc: 0.9885 | Loss: 0.0086 | Patience: 4/5
✅ Epoch  9 | Train Acc: 0.9982 | Val Acc: 0.9922 | Loss: 0.0041 | ✓ NEW BEST
   Epoch 10 | Train Acc: 0.9987 | Val Acc: 0.9906 | Loss: 0.0046 | Patience: 1/5
   Epoch 11 | Train Acc: 0.9988 | Val Acc: 0.9885 | Loss: 0.0052 | Patience: 2/5
   Epoch 12 | Train Acc: 0.9980 | Val Acc: 0.9896 | Loss: 0.0048 | Patience: 3/5
   Epoch 13 | Train Acc:

In [49]:
# Cell 5: Final evaluation and testing
# Load best model
classifier.load_state_dict(torch.load("/content/plastic_classifier_best.pth"))
classifier.eval()

# Make predictions on validation set
all_preds = []
all_probs = []
all_true = []

with torch.no_grad():
    for i in range(0, len(val_features), 32):
        batch_features = val_features[i:i+32].to(device)
        batch_labels = val_labels_tensor[i:i+32]
        outputs = classifier(batch_features).squeeze()
        probs = torch.sigmoid(outputs)
        predictions = (probs > 0.5).float()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(predictions.cpu().numpy())
        all_true.extend(batch_labels.numpy())

# Print detailed metrics
print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)

print("\n📊 Classification Report:")
print(classification_report(all_true, all_preds, target_names=['Non-Plastic', 'Plastic']))

print("\n📊 Confusion Matrix:")
cm = confusion_matrix(all_true, all_preds)
print(cm)
print(f"\nTrue Negatives (Non-Plastic correct): {cm[0,0]}")
print(f"False Positives (Non-Plastic wrong): {cm[0,1]}")
print(f"False Negatives (Plastic wrong): {cm[1,0]}")
print(f"True Positives (Plastic correct): {cm[1,1]}")

# Calculate additional metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(f"\n📈 Detailed Metrics:")
print(f"Accuracy:  {accuracy_score(all_true, all_preds):.4f}")
print(f"Precision: {precision_score(all_true, all_preds):.4f}")
print(f"Recall:    {recall_score(all_true, all_preds):.4f}")
print(f"F1-Score:  {f1_score(all_true, all_preds):.4f}")

# Test on a few sample images
print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

def predict_plastic(image_path):
    """Predict if an image contains plastic"""
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device)

    with torch.no_grad():
        features = dino_model(pixel_values).last_hidden_state[:, 0, :]
        output = classifier(features)
        probability = torch.sigmoid(output).item()

    return probability

# Test on some validation images
for i in range(min(10, len(val_images))):
    prob = predict_plastic(val_images[i])
    prediction = "PLASTIC" if prob > 0.5 else "NON-PLASTIC"
    actual = "PLASTIC" if val_labels[i] == 1 else "NON-PLASTIC"
    confidence = prob if prob > 0.5 else 1-prob
    print(f"{i+1:2d}. {prediction:10s} (conf: {confidence:.3f}) | Actual: {actual:10s} | {'✓' if prediction == actual else '✗'}")


FINAL MODEL PERFORMANCE

📊 Classification Report:
              precision    recall  f1-score   support

 Non-Plastic       1.00      0.99      0.99       959
     Plastic       0.99      1.00      0.99       958

    accuracy                           0.99      1917
   macro avg       0.99      0.99      0.99      1917
weighted avg       0.99      0.99      0.99      1917


📊 Confusion Matrix:
[[947  12]
 [  3 955]]

True Negatives (Non-Plastic correct): 947
False Positives (Non-Plastic wrong): 12
False Negatives (Plastic wrong): 3
True Positives (Plastic correct): 955

📈 Detailed Metrics:
Accuracy:  0.9922
Precision: 0.9876
Recall:    0.9969
F1-Score:  0.9922

SAMPLE PREDICTIONS
 1. PLASTIC    (conf: 0.755) | Actual: NON-PLASTIC | ✗
 2. NON-PLASTIC (conf: 1.000) | Actual: NON-PLASTIC | ✓
 3. PLASTIC    (conf: 0.998) | Actual: PLASTIC    | ✓
 4. PLASTIC    (conf: 1.000) | Actual: PLASTIC    | ✓
 5. PLASTIC    (conf: 1.000) | Actual: PLASTIC    | ✓
 6. NON-PLASTIC (conf: 1.000) | Actu